In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ishanjha100/student-passfail-data")

print("Path to dataset files:", path)

100%|██████████| 930/930 [00:00<00:00, 748kB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/ishanjha100/student-passfail-data/versions/1


In [3]:
import os
import kagglehub
DATA_ROOT = kagglehub.dataset_download("ishanjha100/student-passfail-data")
print('DATA_ROOT:', DATA_ROOT)
for root, dirs, files in os.walk(DATA_ROOT):
    print(root, '| files:', len(files), '|', files[:3])

Using Colab cache for faster access to the 'student-passfail-data' dataset.
DATA_ROOT: /kaggle/input/student-passfail-data
/kaggle/input/student-passfail-data | files: 2 | ['Pass-Fail Data.csv', '.nfs00000000077695e200000079']


In [4]:
#read dataframe
df = pd.read_csv(os.path.join(DATA_ROOT, 'Pass-Fail Data.csv'))
df.head()

,student_id,attendance_pct,homework_pct,midterm_score,study_hours_per_week,pass
0,1,95,92,88,12,1
1,2,88,85,79,10,1
2,3,60,55,58,4,0
3,4,72,70,65,6,1
4,5,40,45,50,3,0


In [5]:
df=df.drop(columns='student_id',axis=1)

In [6]:
x=df.drop(columns='pass',axis=1)
y=df['pass']

#K-Fold Cross-Validation

In [7]:
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
model=LogisticRegression()
kfold=KFold(n_splits=5,shuffle=True,random_state=42)
cv_results=cross_val_score(model,x,y,cv=kfold,scoring='accuracy')


In [8]:
result=cross_val_score(model,x,y,cv=kfold,scoring='accuracy')
print(result)

[1.   0.95 1.   0.95 1.  ]


In [9]:
print(np.mean(result))

0.9800000000000001


In [10]:
for fold,(train_idx,test_idx) in enumerate(kfold.split(x,y)):
    print(f'Fold:{fold+1}')

Fold:1
Fold:2
Fold:3
Fold:4
Fold:5


In [13]:
print("\nFold Details:")
for fold, (train_idx, test_idx) in enumerate(kfold.split(x), start=1):
    print(f"Fold {fold}:")
    print(f"  Train Indices: {train_idx} -> Data:\n{x.iloc[train_idx].head(2)}")
    print(f"  Test Indices:  {test_idx}  -> Data:\n{x.iloc[test_idx].head(2)}")
    print("\n")


Fold Details:
Fold 1:
  Train Indices: [ 1  2  3  5  6  7  8  9 11 13 14 15 16 17 19 20 21 23 24 25 26 27 28 29
 32 34 35 36 37 38 40 41 42 43 46 47 48 49 50 51 52 54 55 56 57 58 59 60
 61 62 63 64 65 66 67 68 69 71 72 74 75 78 79 81 82 84 85 86 87 88 89 91
 92 93 94 95 96 97 98 99] -> Data:
   attendance_pct  homework_pct  midterm_score  study_hours_per_week
1              88            85             79                    10
2              60            55             58                     4
  Test Indices:  [ 0  4 10 12 18 22 30 31 33 39 44 45 53 70 73 76 77 80 83 90]  -> Data:
   attendance_pct  homework_pct  midterm_score  study_hours_per_week
0              95            92             88                    12
4              40            45             50                     3


Fold 2:
  Train Indices: [ 0  1  2  3  4  6  7  8 10 12 13 14 17 18 19 20 21 22 23 24 25 27 29 30
 31 32 33 34 36 37 38 39 41 43 44 45 46 48 49 50 51 52 53 54 56 57 58 59
 60 61 62 63 64 67 68 70 71 73

#STRATIFIED K-FOLD CROSS-VALIDATION

#kFold create class imbalanced

stratified kfold solve this

In [14]:
from sklearn.model_selection import StratifiedKFold
skfold=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
cv_results=cross_val_score(model,x,y,cv=skfold,scoring='accuracy')

In [25]:
for fold, (train_idx, test_idx) in enumerate(skfold.split(x, y), start=1):
    test_y = y.iloc[test_idx]
    pass_ratio = test_y.mean()
    print(f"Fold {fold} -> Test size: {len(test_idx)}, Pass ratio in test set: {pass_ratio:.2f}")
    print(x.iloc[test_idx].head(2))
    print("\n")

Fold 1 -> Test size: 20, Pass ratio in test set: 0.60
   attendance_pct  homework_pct  midterm_score  study_hours_per_week
0              95            92             88                    12
1              88            85             79                    10


Fold 2 -> Test size: 20, Pass ratio in test set: 0.60
    attendance_pct  homework_pct  midterm_score  study_hours_per_week
6               78            80             74                     8
12              74            70             68                     7


Fold 3 -> Test size: 20, Pass ratio in test set: 0.60
   attendance_pct  homework_pct  midterm_score  study_hours_per_week
4              40            45             50                     3
9              68            65             60                     6


Fold 4 -> Test size: 20, Pass ratio in test set: 0.60
   attendance_pct  homework_pct  midterm_score  study_hours_per_week
2              60            55             58                     4
3              7

# 3. LEAVE-ONE-OUT CROSS-VALIDATION (LOO)

In [18]:
from sklearn.model_selection import LeaveOneOut
loo=LeaveOneOut()
skfold=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
cv_results=cross_val_score(model,x,y,cv=loo,scoring='accuracy')

In [22]:
for fold, (train_idx, test_idx) in enumerate(loo.split(x, y), start=1):
    if fold <= 5:
        print(f"Iteration {fold} -> Test sample index: {test_idx[0]}")
        print(x.iloc[test_idx])

Iteration 1 -> Test sample index: 0
   attendance_pct  homework_pct  midterm_score  study_hours_per_week
0              95            92             88                    12
Iteration 2 -> Test sample index: 1
   attendance_pct  homework_pct  midterm_score  study_hours_per_week
1              88            85             79                    10
Iteration 3 -> Test sample index: 2
   attendance_pct  homework_pct  midterm_score  study_hours_per_week
2              60            55             58                     4
Iteration 4 -> Test sample index: 3
   attendance_pct  homework_pct  midterm_score  study_hours_per_week
3              72            70             65                     6
Iteration 5 -> Test sample index: 4
   attendance_pct  homework_pct  midterm_score  study_hours_per_week
4              40            45             50                     3


# 4. GROUP K-FOLD CROSS-VALIDATION

In [27]:
np.random.seed(42)
groups = np.repeat(np.arange(5), len(df) // 5)
print(df)

    attendance_pct  homework_pct  midterm_score  study_hours_per_week  pass
0               95            92             88                    12     1
1               88            85             79                    10     1
2               60            55             58                     4     0
3               72            70             65                     6     1
4               40            45             50                     3     0
..             ...           ...            ...                   ...   ...
95              89            92             94                    14     1
96              79            77             75                     8     1
97              55            56             58                     5     0
98              91            90             88                    12     1
99              52            50             53                     4     0

[100 rows x 5 columns]


In [29]:
from sklearn.model_selection import GroupKFold
gkf = GroupKFold(n_splits=5)
gkf_scores = cross_val_score(model, x, y, cv=gkf, groups=groups, scoring='accuracy')

In [31]:
for fold, (train_idx, test_idx) in enumerate(gkf.split(x, y, groups=groups), start=1):
    held_out_groups = np.unique(groups[test_idx])
    print(f"Fold {fold} -> Held-out Test Group(s): {held_out_groups}")


Fold 1 -> Held-out Test Group(s): [4]
Fold 2 -> Held-out Test Group(s): [3]
Fold 3 -> Held-out Test Group(s): [2]
Fold 4 -> Held-out Test Group(s): [1]
Fold 5 -> Held-out Test Group(s): [0]


# 4. GROUP K-FOLD CROSS-VALIDATION

একটি হাসপাতালে ৫ জন রোগীর (A, B, C, D, E) প্রত্যেকের ৪টি করে মোট ২০টি এক্স-রে ছবি আছে।

Fold 1: Patient A, B, C, D-এর সব ছবি (১৬টি) দিয়ে মডেল ট্রেন হবে এবং Patient E-এর সব ছবি (৪টি) দিয়ে মডেল টেস্ট হবে।

Patient E-এর কোনো ছবিই ট্রেইনিংয়ে থাকবে না। এতে বোঝা যাবে মডেলটি সম্পূর্ণ নতুন একজন অচেনা রোগীর এক্স-রে সঠিকভাবে শনাক্ত করতে পারছে কিনা।

In [32]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold


X = np.array([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10], [11, 12]])
y = np.array([1, 1, 0, 0, 1, 0])

In [34]:
print(X)

[[ 1  2]
 [ 3  4]
 [ 5  6]
 [ 7  8]
 [ 9 10]
 [11 12]]


In [35]:
groups = np.array([101, 101, 102, 102, 103, 103])

In [36]:
gkf = GroupKFold(n_splits=3)


for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    print(f"--- Fold {fold} ---")
    print(f"Train Groups : {groups[train_idx]}")
    print(f"Test Group   : {groups[test_idx]}")
    print(f"Train Indices: {train_idx}")
    print(f"Test Indices : {test_idx}\n")

--- Fold 1 ---
Train Groups : [101 101 102 102]
Test Group   : [103 103]
Train Indices: [0 1 2 3]
Test Indices : [4 5]

--- Fold 2 ---
Train Groups : [101 101 103 103]
Test Group   : [102 102]
Train Indices: [0 1 4 5]
Test Indices : [2 3]

--- Fold 3 ---
Train Groups : [102 102 103 103]
Test Group   : [101 101]
Train Indices: [2 3 4 5]
Test Indices : [0 1]



#Time Series K-Fold / TimeSeriesSplit

In [37]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sudalairajkumar/daily-temperature-of-major-cities")

print("Path to dataset files:", path)

100%|██████████| 12.9M/12.9M [00:00<00:00, 142MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/sudalairajkumar/daily-temperature-of-major-cities/versions/1


In [39]:
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

/tmp/ipykernel_1429/680804159.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(path, csv_file))


In [56]:
f = df[(df['AvgTemperature'] != -99.0) & (df['Day'] > 0)].copy()

In [57]:
df['City'] = df['City'].astype(str).str.strip()
city_df = df[df['City'].str.lower() == 'tokyo'].copy()

In [58]:
city_df['Date'] = pd.to_datetime(city_df[['Year', 'Month', 'Day']], errors='coerce')
city_df = city_df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

In [43]:
city_df['Temp_Lag1'] = city_df['AvgTemperature'].shift(1)
city_df['Temp_Lag2'] = city_df['AvgTemperature'].shift(2)
city_df['Temp_Lag7'] = city_df['AvgTemperature'].shift(7)

In [44]:
print(city_df['Temp_Lag1'])

0        NaN
1       43.0
2       40.8
3       43.3
4       44.9
        ... 
9232    64.3
9233    67.1
9234    71.1
9235    70.9
9236    71.5
Name: Temp_Lag1, Length: 9237, dtype: float64


In [45]:
city_df = city_df.dropna().reset_index(drop=True)

In [46]:
print(city_df)

Empty DataFrame
Columns: [Region, Country, State, City, Month, Day, Year, AvgTemperature, Date, Temp_Lag1, Temp_Lag2, Temp_Lag7]
Index: []


In [47]:
X = city_df[['Temp_Lag1', 'Temp_Lag2', 'Temp_Lag7']]
y = city_df['AvgTemperature']
dates = city_df['Date']

In [55]:
print(f"X- sample (Shape): {X.shape}")

X- sample (Shape): (0, 3)


In [61]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge
n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)
model=Ridge()


--- 5-Fold Time Series Split ---


ValueError: Cannot have number of folds=6 greater than the number of samples=0.

In [65]:
import os
import kagglehub
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

# 1. Download Dataset
path = kagglehub.dataset_download(
    "sudalairajkumar/daily-temperature-of-major-cities"
)

csv_file = [
    f for f in os.listdir(path) if f.endswith('.csv')
][0]

df = pd.read_csv(os.path.join(path, csv_file))

# 2. Clean Column Names
df.columns = df.columns.str.strip()

# 3. Check Required Columns
required_columns = [
    'City', 'Year', 'Month', 'Day', 'AvgTemperature'
]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Missing column: {col}")

# 4. Convert Numeric Columns
for col in ['Year', 'Month', 'Day', 'AvgTemperature']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 5. Clean Data
df['City'] = df['City'].astype(str).str.strip()

df = df.dropna(
    subset=['City', 'Year', 'Month', 'Day', 'AvgTemperature']
)

df = df[
    (df['AvgTemperature'] != -99.0) &
    (df['Day'] > 0)
].copy()

print("Dataset after cleaning:", df.shape)

# 6. Select Tokyo
city_df = df[
    df['City'].str.casefold() == 'tokyo'
].copy()

print("Tokyo records:", len(city_df))

if city_df.empty:
    print("Available Cities:")
    print(df['City'].unique()[:100])
    raise ValueError("Tokyo city not found in dataset!")

# 7. Create Date
city_df['Date'] = pd.to_datetime(
    {
        'year': city_df['Year'],
        'month': city_df['Month'],
        'day': city_df['Day']
    },
    errors='coerce'
)

city_df = city_df.dropna(subset=['Date'])

city_df = city_df.sort_values('Date').reset_index(drop=True)

# 8. Remove Duplicate Dates
city_df = city_df.drop_duplicates(
    subset=['Date'],
    keep='last'
).reset_index(drop=True)

print("Valid Tokyo records:", len(city_df))

# 9. Create Lag Features
city_df['Temp_Lag1'] = city_df['AvgTemperature'].shift(1)
city_df['Temp_Lag2'] = city_df['AvgTemperature'].shift(2)
city_df['Temp_Lag7'] = city_df['AvgTemperature'].shift(7)

# FIX: Only drop rows where lag features are NaN, avoiding columns like 'State'
city_df = city_df.dropna(subset=['AvgTemperature', 'Temp_Lag1', 'Temp_Lag2', 'Temp_Lag7']).reset_index(drop=True)

# 10. Prepare X and y
X = city_df[
    ['Temp_Lag1', 'Temp_Lag2', 'Temp_Lag7']
]

y = city_df['AvgTemperature']

print("\nX Shape:", X.shape)
print("Y Shape:", y.shape)

# 11. Time Series Split
n_splits = 5

if len(X) <= n_splits:
    raise ValueError(
        f"Not enough samples: {len(X)}. "
        "At least 6 samples are required for 5 splits."
    )

tscv = TimeSeriesSplit(n_splits=n_splits)

model = Ridge(alpha=1.0)

print(f"\n--- {n_splits}-Fold Time Series Split ---")

# 12. Model Training and Evaluation
rmse_scores = []

for fold, (train_idx, test_idx) in enumerate(
    tscv.split(X), start=1
):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    rmse = np.sqrt(
        mean_squared_error(y_test, preds)
    )

    rmse_scores.append(rmse)

    print(
        f"Fold {fold}: "
        f"Train Size = {len(train_idx)}, "
        f"Test Size = {len(test_idx)}, "
        f"RMSE = {rmse:.4f}"
    )

# 13. Average RMSE
print("\nAverage RMSE:", np.mean(rmse_scores))

Using Colab cache for faster access to the 'daily-temperature-of-major-cities' dataset.


/tmp/ipykernel_1429/1634702154.py:19: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(path, csv_file))


Dataset after cleaning: (2826655, 8)
Tokyo records: 9237
Valid Tokyo records: 9237

X Shape: (9230, 3)
Y Shape: (9230,)

--- 5-Fold Time Series Split ---
Fold 1: Train Size = 1540, Test Size = 1538, RMSE = 3.4630
Fold 2: Train Size = 3078, Test Size = 1538, RMSE = 3.7531
Fold 3: Train Size = 4616, Test Size = 1538, RMSE = 3.5974
Fold 4: Train Size = 6154, Test Size = 1538, RMSE = 3.4134
Fold 5: Train Size = 7692, Test Size = 1538, RMSE = 3.8028

Average RMSE: 3.605931628056838
